## RGB_Adapter
Convert pictures with four color channels into RGB bands.

In [1]:
from osgeo import gdal
import numpy as np
import os

In [ ]:
def convert_to_rgb(input_file, output_file):
    dataset = gdal.Open(input_file)
    
    if dataset.RasterCount >= 3: 
        
        band1 = dataset.GetRasterBand(1).ReadAsArray()  # Red
        band2 = dataset.GetRasterBand(2).ReadAsArray()  # Green
        band3 = dataset.GetRasterBand(3).ReadAsArray()  # Blue
        
        if dataset.RasterCount == 4:
            band4 = dataset.GetRasterBand(4).ReadAsArray()  # Undefined (ignore or process accordingly)
        
        r = band1.astype(np.uint8) 
        g = band2.astype(np.uint8)
        b = band3.astype(np.uint8)
        
        # Create an RGB image
        rgb_image = np.dstack((r, g, b))

        driver = gdal.GetDriverByName("GTiff")
        out_ds = driver.Create(output_file, dataset.RasterXSize, dataset.RasterYSize, 3, gdal.GDT_Byte)

        out_ds.GetRasterBand(1).WriteArray(r)
        out_ds.GetRasterBand(2).WriteArray(g)
        out_ds.GetRasterBand(3).WriteArray(b)

        out_ds.SetGeoTransform(dataset.GetGeoTransform())
        out_ds.SetProjection(dataset.GetProjection())

        out_ds.FlushCache()
        out_ds = None
        dataset = None
        print(f"Converted {input_file} to RGB and saved as {output_file}")
    else:
        print(f"{input_file} does not have enough bands for RGB conversion")


In [ ]:
## test
path = "/Volumes/Elements/Other Sources/SCOOP2018UnComp_TorontoBoundarywgs84/"
img_files = [f for f in os.listdir(path) if f.endswith('.tif') and not f.startswith('.')]

for file in img_files:
    input_file = f"/Volumes/Elements/Other Sources/SCOOP2018UnComp_TorontoBoundarywgs84/{file}"
    output_file = f"/Volumes/Elements/Other Sources/TorontoBoundarywgs84_adj/{file}"
    convert_to_rgb(input_file,output_file)